In [67]:
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder, RobustScaler

In [68]:
DATA_PATH = Path("E:\\download\\project_cs114\\data\\raw\\Crop_recommendation.csv")

if not DATA_PATH.exists():
    raise FileNotFoundError(f"Missing data file: {DATA_PATH}")

df = pd.read_csv(DATA_PATH)
df.head()

,N,P,K,temperature,humidity,ph,rainfall,label
0,90,42,43,20.879744,82.002744,6.502985,202.935536,rice
1,85,58,41,21.770462,80.319644,7.038096,226.655537,rice
2,60,55,44,23.004459,82.320763,7.840207,263.964248,rice
3,74,35,40,26.491096,80.158363,6.980401,242.864034,rice
4,78,42,42,20.130175,81.604873,7.628473,262.717340,rice


In [69]:
REQUIRED_COLUMNS = ["N", "P", "K", "temperature", "humidity", "ph", "rainfall"]
TARGET_COLUMN = "label"
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2200 entries, 0 to 2199
Data columns (total 8 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   N            2200 non-null   int64  
 1   P            2200 non-null   int64  
 2   K            2200 non-null   int64  
 3   temperature  2200 non-null   float64
 4   humidity     2200 non-null   float64
 5   ph           2200 non-null   float64
 6   rainfall     2200 non-null   float64
 7   label        2200 non-null   object 
dtypes: float64(4), int64(3), object(1)
memory usage: 137.6+ KB


In [70]:
df[REQUIRED_COLUMNS].describe().T

,count,mean,std,min,25%,50%,75%,max
N,2200.0,50.551818,36.917334,0.000000,21.000000,37.000000,84.250000,140.000000
P,2200.0,53.362727,32.985883,5.000000,28.000000,51.000000,68.000000,145.000000
K,2200.0,48.149091,50.647931,5.000000,20.000000,32.000000,49.000000,205.000000
temperature,2200.0,25.616244,5.063749,8.825675,22.769375,25.598693,28.561654,43.675493
humidity,2200.0,71.481779,22.263812,14.258040,60.261953,80.473146,89.948771,99.981876
ph,2200.0,6.469480,0.773938,3.504752,5.971693,6.425045,6.923643,9.935091
rainfall,2200.0,103.463655,54.958389,20.211267,64.551686,94.867624,124.267508,298.560117


In [71]:
y = df[TARGET_COLUMN]
print("Number of classes:", y.nunique())
print("Classes:\n", sorted(y.unique()))

y.value_counts().head(10)

Number of classes: 22
Classes:
 ['apple', 'banana', 'blackgram', 'chickpea', 'coconut', 'coffee', 'cotton', 'grapes', 'jute', 'kidneybeans', 'lentil', 'maize', 'mango', 'mothbeans', 'mungbean', 'muskmelon', 'orange', 'papaya', 'pigeonpeas', 'pomegranate', 'rice', 'watermelon']


label
rice           100
maize          100
chickpea       100
kidneybeans    100
pigeonpeas     100
mothbeans      100
mungbean       100
blackgram      100
lentil         100
pomegranate    100
Name: count, dtype: int64

In [72]:
class ColumnSchemaEnforcer(BaseEstimator, TransformerMixin):
    def __init__(self, required_columns, drop_extra=True):
        self.required_columns = list(required_columns)
        self.drop_extra = drop_extra

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()
        missing = [c for c in self.required_columns if c not in X.columns]
        if missing:
            raise ValueError(f"Missing columns: {missing}")
        if self.drop_extra:
            X = X[self.required_columns]
        else:
            extra = [c for c in X.columns if c not in self.required_columns]
            X = X[self.required_columns + extra]
        return X


class TypeCaster(BaseEstimator, TransformerMixin):
    def __init__(self, int_cols, float_cols):
        self.int_cols = list(int_cols)
        self.float_cols = list(float_cols)

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()
        for col in self.int_cols + self.float_cols:
            X[col] = pd.to_numeric(X[col], errors="coerce")
        return X


class QuantileClipper(BaseEstimator, TransformerMixin):
    def __init__(self, lower=0.01, upper=0.99):
        self.lower = lower
        self.upper = upper

    def fit(self, X, y=None):
        if isinstance(X, pd.DataFrame):
            self.feature_names_in_ = list(X.columns)
            Xv = X.to_numpy(dtype=float)
        else:
            self.feature_names_in_ = None
            Xv = np.asarray(X, dtype=float)

        self.lower_bounds_ = np.nanquantile(Xv, self.lower, axis=0)
        self.upper_bounds_ = np.nanquantile(Xv, self.upper, axis=0)
        return self

    def transform(self, X):
        if isinstance(X, pd.DataFrame):
            Xv = X.to_numpy(dtype=float)
            clipped = np.clip(Xv, self.lower_bounds_, self.upper_bounds_)
            return pd.DataFrame(clipped, columns=list(X.columns), index=X.index)

        Xv = np.asarray(X, dtype=float)
        return np.clip(Xv, self.lower_bounds_, self.upper_bounds_)


class FeatureEngineer(BaseEstimator, TransformerMixin):
    def __init__(self, add_ratios=True, add_interactions=True, add_log_rainfall=False, eps=1e-6):
        self.add_ratios = add_ratios
        self.add_interactions = add_interactions
        self.add_log_rainfall = add_log_rainfall
        self.eps = eps

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()
        if {"N", "P", "K"}.issubset(X.columns):
            X["total_npk"] = X["N"] + X["P"] + X["K"]
            if self.add_ratios:
                X["ratio_n_p"] = X["N"] / (X["P"] + self.eps)
                X["ratio_n_k"] = X["N"] / (X["K"] + self.eps)
                X["ratio_p_k"] = X["P"] / (X["K"] + self.eps)
        if self.add_interactions and {"temperature", "humidity"}.issubset(X.columns):
            X["temp_humidity"] = X["temperature"] * X["humidity"]
        if self.add_log_rainfall and "rainfall" in X.columns:
            X["log_rainfall"] = np.log1p(X["rainfall"].clip(lower=0))
        return X

In [73]:
use_feature_engineering = True

clipper = QuantileClipper(lower=0.01, upper=0.99)
scaler = RobustScaler()
feature_engineer = FeatureEngineer(add_ratios=True, add_interactions=True, add_log_rainfall=True, eps=1e-6)

preprocessor = Pipeline(
    steps=[
        ("schema", ColumnSchemaEnforcer(REQUIRED_COLUMNS, drop_extra=True)),
        ("types", TypeCaster(int_cols=["N", "P", "K"], float_cols=["temperature", "humidity", "ph", "rainfall"])),
        ("features", feature_engineer if use_feature_engineering else "passthrough"),
        ("impute", SimpleImputer(strategy="median")),
        ("clip", clipper),
        ("scale", scaler),
    ]
)

preprocessor

,steps,"[('schema', ...), ('types', ...), ...]"
,transform_input,None
,memory,None
,verbose,False
,required_columns,"['N', 'P', ...]"
,drop_extra,True
,int_cols,"['N', 'P', ...]"
,float_cols,"['temperature', 'humidity', ...]"
,add_ratios,True
,add_interactions,True
,add_log_rainfall,True


In [74]:
X = df[REQUIRED_COLUMNS]
y = df[TARGET_COLUMN]

X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42,
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.5,
    stratify=y_temp,
    random_state=42,
)

X_train_prep = preprocessor.fit_transform(X_train)
X_val_prep = preprocessor.transform(X_val)
X_test_prep = preprocessor.transform(X_test)

print("Train shape:", X_train_prep.shape)
print("Val shape:", X_val_prep.shape)
print("Test shape:", X_test_prep.shape)

label_encoder = LabelEncoder()
y_train_enc = label_encoder.fit_transform(y_train)
y_val_enc = label_encoder.transform(y_val)
y_test_enc = label_encoder.transform(y_test)

print("Encoded labels example:", y_train_enc[:10])

Train shape: (1760, 13)
Val shape: (220, 13)
Test shape: (220, 13)
Encoded labels example: [16  7  9 13 16  6  1 16  7 10]


In [75]:
def build_feature_names(base_cols, use_engineering):
    names = list(base_cols)
    if use_engineering:
        names += [
            "total_npk",
            "ratio_n_p",
            "ratio_n_k",
            "ratio_p_k",
            "temp_humidity",
            "log_rainfall",
        ]
    return names

nan_count = np.isnan(X_train_prep).sum()
inf_count = np.isinf(X_train_prep).sum()
print("NaN count after preprocessing:", nan_count)
print("Inf count after preprocessing:", inf_count)

feature_names = build_feature_names(REQUIRED_COLUMNS, use_feature_engineering)
X_train_clipped_df = pd.DataFrame(X_train_prep, columns=feature_names)

print("\nClipped data summary (train):")
X_train_clipped_df.describe().T

NaN count after preprocessing: 0
Inf count after preprocessing: 0

Clipped data summary (train):


,count,mean,std,min,25%,50%,75%,max
N,1760.0,0.210672,0.573762,-0.578125,-0.250000,0.000000e+00,0.750000,1.409844
P,1760.0,0.058437,0.822520,-1.125000,-0.575000,0.000000e+00,0.425000,2.300000
K,1760.0,0.557026,1.747285,-0.841724,-0.413793,0.000000e+00,0.586207,5.931034
temperature,1760.0,0.015813,0.867227,-2.375617,-0.474394,0.000000e+00,0.525606,2.727339
humidity,1760.0,-0.307459,0.748740,-2.198392,-0.684782,0.000000e+00,0.315218,0.552164
ph,1760.0,0.043348,0.773770,-1.888412,-0.487344,-4.551210e-16,0.512656,2.385080
rainfall,1760.0,0.144820,0.923709,-1.234668,-0.511868,0.000000e+00,0.488132,2.972171
total_npk,1760.0,0.072139,0.951364,-1.357143,-0.619048,0.000000e+00,0.380952,2.738095
ratio_n_p,1760.0,0.456858,1.348119,-0.541353,-0.328067,-3.366448e-17,0.671933,7.666096
ratio_n_k,1760.0,0.158386,0.903189,-0.865410,-0.521993,-6.862843e-17,0.478007,3.554992


In [76]:
feature_names = build_feature_names(REQUIRED_COLUMNS, use_feature_engineering)

X_train_prep_df = pd.DataFrame(X_train_prep, columns=feature_names)
X_val_prep_df = pd.DataFrame(X_val_prep, columns=feature_names)
X_test_prep_df = pd.DataFrame(X_test_prep, columns=feature_names)

train_with_y = X_train_prep_df.copy()
train_with_y[TARGET_COLUMN] = y_train_enc.copy()

val_with_y = X_val_prep_df.copy()
val_with_y[TARGET_COLUMN] = y_val_enc.copy()

test_with_y = X_test_prep_df.copy()
test_with_y[TARGET_COLUMN] = y_test_enc.copy()

train_with_y.to_csv("E:\\download\\project_cs114\\data\\processed\\train_preprocessed.csv", index=False)
val_with_y.to_csv("E:\\download\\project_cs114\\data\\processed\\val_preprocessed.csv", index=False)
test_with_y.to_csv("E:\\download\\project_cs114\\data\\processed\\test_preprocessed.csv", index=False)